[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/100.%20The%20Closed-Loop%20%28Reverse%20Logistics%29%20Network%20Design%20Problem/P100-Tier-10_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 100. The Closed-Loop Network Design Problem
## Tier 10: Proactive Demand and Market Shaping

### Purpose and outputs

This example represents a market shaper that examines customer segments, evaluates strategic interventions under a budget, simulates market-share changes over a planning horizon, and reports the resulting indicators. Its strategy search enumerates feasible intervention combinations and evaluates each with a stochastic simulation.

The inputs are the segment characteristics, intervention costs and impacts, a budget, and a time horizon. The outputs include a selected intervention set, simulated segment market shares, and summary indicators. The code is an illustrative model, not a calibrated market forecast.

### Model assumptions and scope

- Each segment has its own market share. Segment sizes are used to convert those shares into customer equivalents; the segments are treated as distinct groups.
- An intervention becomes active when its implementation time has elapsed. In each active month, it has an independent chance of success. A successful month adds its specified market-share impact, scaled by 0.01, to each target segment.
- Market shares are capped at 1.0. The simulation does not model competitor shares, interactions between interventions, changing segment sizes, or customer movement between segments.
- The model uses an illustrative value of 1,000 monetary units per net additional customer equivalent to calculate an approximate ROI. This is a modeling assumption, not an observed revenue figure or a validated customer lifetime value.
- Segment sensitivities and network configuration are recorded, but the simulation does not use them to derive intervention impacts. Those impacts are supplied directly as intervention inputs.
- The strategy search evaluates combinations of up to four interventions. It does not guarantee that the best combination among larger sets is considered.

### Solution sequence

1. Define customer segments, interventions, and the market-shaping system.
2. Summarize market opportunities from the segment data.
3. Enumerate intervention combinations that fit the budget and simulate each combination.
4. Select the combination with the highest score and inspect its simulated outcomes.
5. Visualize the segment, investment, and simulation results.

### Mathematical model

Let \(S\) be the set of customer segments and \(I\) the set of interventions. For a selected intervention set \(A \subseteq I\), feasibility requires

$$\sum_{i \in A} c_i \leq B,$$

where \(c_i\) is intervention \(i\)'s investment cost and \(B\) is the budget. The search considers sets with at most four interventions. This is exhaustive subset enumeration over the specified sizes, implemented by recursive combination generation.

For segment \(s\), let \(q_{s,t}\) be its market share at month \(t\). For an active intervention \(i\) targeting \(s\), let \(p_i\) be its monthly success probability, \(m_i\) its market-share impact, and \(\xi_{i,t}\) a Bernoulli random variable that is 1 on success and 0 otherwise. The update is

$$q_{s,t+1} = \min\!\left(1,\ q_{s,t} + 0.01\sum_{i \in A_s} m_i\,\xi_{i,t}\right),$$

where \(A_s\) is the set of selected interventions targeting segment \(s\). An intervention is active when the month index is at least its implementation time. The code applies each active intervention in sequence, so the cap is enforced after each successful impact.

Let \(n_s\) be segment size and \(q_{s,0}\) its initial market share. The model's market-growth indicator is the change in customer equivalents relative to the initial customer equivalents:

$$G(A) = \frac{\sum_{s \in S} n_s q_{s,T} - \sum_{s \in S} n_s q_{s,0}}{\sum_{s \in S} n_s q_{s,0}}.$$

Using the illustrative value \(v=1000\) per net additional customer equivalent, approximate ROI is

$$R(A) = \frac{v\left(\sum_{s \in S} n_s q_{s,T} - \sum_{s \in S} n_s q_{s,0}\right)-\sum_{i\in A}c_i}{\sum_{i\in A}c_i}.$$

For a nonempty selected set, the strategy score is \(R(A)+0.5G(A)\). The search retains the empty strategy as a zero-score baseline. When no intervention combination improves on that baseline, the model selects no intervention. Because each candidate is simulated stochastically once, candidate scores can vary with the random-number sequence; the fixed seed makes a complete notebook run reproducible.

If there are \(N\) interventions and the search considers sizes 1 through \(K\), combination generation examines \(\sum_{r=1}^{K}\binom{N}{r}\) candidates, subject to the budget. Each candidate also incurs the cost of its simulation over the time horizon. The recursive helper constructs each combination by choosing intervention indices in increasing order, so it generates each subset once.

The segment response equation in `MarketSegment.calculate_demand_response` is a separate elasticity-based calculation. It describes how modeled attribute changes could affect demand, but the intervention simulation in this example uses the directly supplied `expected_impact` values instead.

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass, field
from enum import Enum
from itertools import combinations
import random
import time
from typing import Any, Dict, List, Optional, Set, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fixing both random generators makes the stochastic simulation reproducible.
np.random.seed(42)
random.seed(42)

In [ ]:
class CustomerSegment(Enum):
    """Customer groups used to organize market characteristics."""

    PRICE_SENSITIVE = "price_sensitive"
    QUALITY_FOCUSED = "quality_focused"
    CONVENIENCE_SEEKING = "convenience_seeking"
    SUSTAINABILITY_CONSCIOUS = "sustainability_conscious"
    PREMIUM_SEGMENT = "premium_segment"


@dataclass
class MarketSegment:
    """Market segment data and an elasticity-based response calculation."""

    segment_id: str
    segment_type: CustomerSegment
    size: int
    price_elasticity: float
    quality_sensitivity: float
    convenience_sensitivity: float
    sustainability_sensitivity: float
    current_market_share: float = 0.0
    growth_potential: float = 0.0

    def calculate_demand_response(
        self,
        price_change: float,
        quality_change: float,
        convenience_change: float,
        sustainability_change: float,
    ) -> float:
        """Estimate a segment's response to changes in modeled attributes."""
        price_response = -self.price_elasticity * price_change
        quality_response = self.quality_sensitivity * quality_change
        convenience_response = self.convenience_sensitivity * convenience_change
        sustainability_response = (
            self.sustainability_sensitivity * sustainability_change
        )

        total_response = (
            price_response
            + quality_response
            + convenience_response
            + sustainability_response
        )
        response_multiplier = 1.0 + self.growth_potential * 0.5
        return total_response * response_multiplier


@dataclass
class StrategicIntervention:
    """A market intervention with targets, cost, timing, and impact assumptions."""

    intervention_id: str
    intervention_type: str
    target_segments: List[str]
    investment_cost: float
    expected_impact: Dict[str, float]
    implementation_time: int
    success_probability: float = 0.7

    def calculate_roi(self, market_response: Dict[str, float]) -> float:
        """Calculate the illustrative ROI from supplied market-response values."""
        if self.investment_cost <= 0:
            raise ValueError("Investment cost must be positive to calculate ROI.")

        total_impact = sum(market_response.values())
        return (total_impact * 1000 - self.investment_cost) / self.investment_cost


@dataclass
class MarketShaper:
    """Analyze segments and evaluate intervention combinations by simulation."""

    shaper_id: str
    market_segments: List[MarketSegment] = field(default_factory=list)
    interventions: List[StrategicIntervention] = field(default_factory=list)
    network_configuration: Dict[str, Any] = field(default_factory=dict)
    market_history: List[Dict[str, Any]] = field(default_factory=list)

    def add_market_segment(self, segment: MarketSegment) -> None:
        """Add a segment after checking its basic model inputs."""
        if segment.size < 0:
            raise ValueError("Segment size cannot be negative.")
        if not 0.0 <= segment.current_market_share <= 1.0:
            raise ValueError("Market share must be between 0 and 1.")
        if segment.growth_potential < 0.0:
            raise ValueError("Growth potential cannot be negative.")
        if any(
            existing.segment_id == segment.segment_id
            for existing in self.market_segments
        ):
            raise ValueError(f"Duplicate segment ID: {segment.segment_id}")
        self.market_segments.append(segment)

    def add_intervention(self, intervention: StrategicIntervention) -> None:
        """Add an intervention after checking inputs used by the simulation."""
        if intervention.investment_cost <= 0:
            raise ValueError("Investment cost must be positive.")
        if intervention.implementation_time < 0:
            raise ValueError("Implementation time cannot be negative.")
        if not 0.0 <= intervention.success_probability <= 1.0:
            raise ValueError("Success probability must be between 0 and 1.")
        if any(
            existing.intervention_id == intervention.intervention_id
            for existing in self.interventions
        ):
            raise ValueError(
                f"Duplicate intervention ID: {intervention.intervention_id}"
            )
        unknown_targets = set(intervention.target_segments) - {
            segment.segment_id for segment in self.market_segments
        }
        if unknown_targets:
            raise ValueError(f"Unknown target segments: {sorted(unknown_targets)}")
        self.interventions.append(intervention)

    def configure_network(self, configuration: Dict[str, Any]) -> None:
        """Store a network description for reporting and future model extensions."""
        self.network_configuration = configuration.copy()

    def analyze_market_opportunities(self) -> Dict[str, Any]:
        """Summarize segment size, growth potential, and modeled advantages."""
        opportunities = {
            "high_growth_segments": [],
            "underserved_segments": [],
            "competitive_advantages": [],
            "strategic_priorities": [],
            "market_size": sum(segment.size for segment in self.market_segments),
            "growth_potential": 0.0,
            "current_market_position": 0.0,
        }

        total_market_size = opportunities["market_size"]
        if total_market_size > 0:
            opportunities["current_market_position"] = sum(
                segment.size * segment.current_market_share
                for segment in self.market_segments
            ) / total_market_size

        for segment in self.market_segments:
            if segment.growth_potential > 0.3:
                opportunities["high_growth_segments"].append(
                    {
                        "segment_id": segment.segment_id,
                        "growth_potential": segment.growth_potential,
                        "size": segment.size,
                    }
                )

            if segment.current_market_share < 0.2:
                opportunities["underserved_segments"].append(
                    {
                        "segment_id": segment.segment_id,
                        "current_share": segment.current_market_share,
                        "size": segment.size,
                    }
                )

            opportunities["growth_potential"] += (
                segment.size * segment.growth_potential
            )

        if any(
            segment.quality_sensitivity > 0.7
            for segment in self.market_segments
        ):
            opportunities["competitive_advantages"].append("quality_leadership")

        if any(
            segment.sustainability_sensitivity > 0.7
            for segment in self.market_segments
        ):
            opportunities["competitive_advantages"].append("sustainability_focus")

        if any(
            segment.convenience_sensitivity > 0.7
            for segment in self.market_segments
        ):
            opportunities["competitive_advantages"].append(
                "convenience_excellence"
            )

        if opportunities["high_growth_segments"]:
            opportunities["strategic_priorities"].append(
                "capture_growth_segments"
            )
        if opportunities["underserved_segments"]:
            opportunities["strategic_priorities"].append(
                "expand_underserved_markets"
            )
        if opportunities["competitive_advantages"]:
            opportunities["strategic_priorities"].append(
                "leverage_competitive_advantages"
            )

        return opportunities

    def simulate_market_response(
        self,
        interventions: List[StrategicIntervention],
        time_horizon: int = 12,
    ) -> Dict[str, Any]:
        """Simulate monthly intervention successes and segment market shares."""
        if time_horizon < 0:
            raise ValueError("Time horizon cannot be negative.")

        known_interventions = {id(item) for item in self.interventions}
        if any(id(item) not in known_interventions for item in interventions):
            raise ValueError("Every simulated intervention must belong to this market shaper.")

        current_shares = {
            segment.segment_id: segment.current_market_share
            for segment in self.market_segments
        }
        initial_customer_equivalents = sum(
            segment.size * segment.current_market_share
            for segment in self.market_segments
        )
        total_market_size = sum(segment.size for segment in self.market_segments)
        simulation_results = {
            "time_horizon": time_horizon,
            "monthly_responses": [],
            "final_market_share": {},
            "total_roi": 0.0,
            "successful_interventions": [],
            "market_growth": 0.0,
        }
        successful_interventions = set()

        for month in range(time_horizon):
            intervention_impacts = defaultdict(float)

            for intervention in interventions:
                if month < intervention.implementation_time:
                    continue

                success = np.random.random() < intervention.success_probability
                if not success:
                    continue

                successful_interventions.add(intervention.intervention_id)
                impact = intervention.expected_impact.get("market_share", 0.0)
                for target_segment_id in intervention.target_segments:
                    if target_segment_id not in current_shares:
                        continue
                    current_shares[target_segment_id] = min(
                        1.0,
                        current_shares[target_segment_id] + impact * 0.01,
                    )
                    intervention_impacts[target_segment_id] += impact

            # Record shares after the month's impacts so each point is an end-of-month state.
            simulation_results["monthly_responses"].append(
                {
                    "month": month + 1,
                    "market_shares": current_shares.copy(),
                    "intervention_impacts": dict(intervention_impacts),
                    "total_market_size": total_market_size,
                }
            )

        final_customer_equivalents = sum(
            segment.size * current_shares[segment.segment_id]
            for segment in self.market_segments
        )
        net_customer_equivalents = (
            final_customer_equivalents - initial_customer_equivalents
        )
        total_investment = sum(
            intervention.investment_cost for intervention in interventions
        )

        if initial_customer_equivalents > 0:
            simulation_results["market_growth"] = (
                net_customer_equivalents / initial_customer_equivalents
            )

        if total_investment > 0:
            modeled_revenue = net_customer_equivalents * 1000
            simulation_results["total_roi"] = (
                modeled_revenue - total_investment
            ) / total_investment

        simulation_results["final_market_share"] = current_shares
        simulation_results["successful_interventions"] = sorted(
            successful_interventions
        )
        return simulation_results

    def optimize_market_strategy(
        self,
        budget: float,
        time_horizon: int = 12,
    ) -> Dict[str, Any]:
        """Enumerate feasible intervention sets and retain the highest score."""
        if budget < 0:
            raise ValueError("Budget cannot be negative.")

        baseline_result = self.simulate_market_response([], time_horizon)
        best_strategy = {
            "selected_interventions": [],
            "expected_roi": 0.0,
            "total_cost": 0.0,
            "market_impact": 0.0,
            "simulation_result": baseline_result,
        }

        max_combination_size = min(len(self.interventions), 4)
        for combination_size in range(1, max_combination_size + 1):
            for combination in self._generate_combinations(
                self.interventions, combination_size
            ):
                total_cost = sum(
                    intervention.investment_cost for intervention in combination
                )
                if total_cost > budget:
                    continue

                simulation_result = self.simulate_market_response(
                    combination, time_horizon
                )
                strategy_score = (
                    simulation_result["total_roi"]
                    + simulation_result["market_growth"] * 0.5
                )

                if strategy_score > best_strategy["expected_roi"]:
                    best_strategy = {
                        "selected_interventions": combination,
                        "expected_roi": strategy_score,
                        "total_cost": total_cost,
                        "market_impact": simulation_result["market_growth"],
                        "simulation_result": simulation_result,
                    }

        return best_strategy

    def _generate_combinations(
        self,
        interventions: List[StrategicIntervention],
        r: int,
    ) -> List[List[StrategicIntervention]]:
        """Generate every size-r subset once using recursive backtracking."""
        if r < 0 or r > len(interventions):
            return []

        generated_combinations = []

        def backtrack(start: int, current_combination: List[StrategicIntervention]) -> None:
            if len(current_combination) == r:
                generated_combinations.append(current_combination.copy())
                return

            for index in range(start, len(interventions)):
                current_combination.append(interventions[index])
                backtrack(index + 1, current_combination)
                current_combination.pop()

        backtrack(0, [])
        return generated_combinations

    def generate_market_report(self) -> Dict[str, Any]:
        """Build a segment-level report from the current market inputs."""
        opportunities = self.analyze_market_opportunities()
        report = {
            "report_timestamp": time.time(),
            "market_overview": {
                "total_segments": len(self.market_segments),
                "total_market_size": opportunities["market_size"],
                "growth_potential": opportunities["growth_potential"],
                "current_market_position": opportunities[
                    "current_market_position"
                ],
            },
            "segment_analysis": {},
            "strategic_opportunities": opportunities,
            "recommended_actions": [],
            "performance_metrics": {},
        }

        for segment in self.market_segments:
            report["segment_analysis"][segment.segment_id] = {
                "type": segment.segment_type.value,
                "size": segment.size,
                "market_share": segment.current_market_share,
                "growth_potential": segment.growth_potential,
                "price_elasticity": segment.price_elasticity,
                "quality_sensitivity": segment.quality_sensitivity,
                "convenience_sensitivity": segment.convenience_sensitivity,
                "sustainability_sensitivity": segment.sustainability_sensitivity,
            }

        if opportunities["high_growth_segments"]:
            report["recommended_actions"].append(
                "Focus on high-growth segments for expansion"
            )
        if opportunities["underserved_segments"]:
            report["recommended_actions"].append(
                "Develop strategies for underserved market segments"
            )
        if "quality_leadership" in opportunities["competitive_advantages"]:
            report["recommended_actions"].append(
                "Leverage quality leadership in market positioning"
            )
        if "sustainability_focus" in opportunities["competitive_advantages"]:
            report["recommended_actions"].append(
                "Emphasize sustainability in market positioning"
            )
        report["recommended_actions"].append(
            "Monitor market response and review intervention assumptions"
        )
        return report

## Build the market instance

The following inputs define five market segments and five candidate interventions. Segment shares are proportions within their respective segments, while intervention impacts are specified as percentage-point-style inputs and scaled by 0.01 during each successful active month. The network configuration is stored as descriptive context; the simulation does not use it to calculate intervention effects.

In [ ]:
print("Initializing the market-shaping system...")
print("=" * 50)

market_shaper = MarketShaper(shaper_id="market_shaper_001")

segments = [
    MarketSegment(
        segment_id="price_sensitive",
        segment_type=CustomerSegment.PRICE_SENSITIVE,
        size=1000,
        price_elasticity=0.8,
        quality_sensitivity=0.3,
        convenience_sensitivity=0.4,
        sustainability_sensitivity=0.2,
        current_market_share=0.15,
        growth_potential=0.2,
    ),
    MarketSegment(
        segment_id="quality_focused",
        segment_type=CustomerSegment.QUALITY_FOCUSED,
        size=800,
        price_elasticity=0.3,
        quality_sensitivity=0.9,
        convenience_sensitivity=0.5,
        sustainability_sensitivity=0.4,
        current_market_share=0.25,
        growth_potential=0.4,
    ),
    MarketSegment(
        segment_id="convenience_seeking",
        segment_type=CustomerSegment.CONVENIENCE_SEEKING,
        size=1200,
        price_elasticity=0.5,
        quality_sensitivity=0.6,
        convenience_sensitivity=0.8,
        sustainability_sensitivity=0.3,
        current_market_share=0.30,
        growth_potential=0.3,
    ),
    MarketSegment(
        segment_id="sustainability_conscious",
        segment_type=CustomerSegment.SUSTAINABILITY_CONSCIOUS,
        size=600,
        price_elasticity=0.4,
        quality_sensitivity=0.7,
        convenience_sensitivity=0.6,
        sustainability_sensitivity=0.9,
        current_market_share=0.20,
        growth_potential=0.5,
    ),
    MarketSegment(
        segment_id="premium_segment",
        segment_type=CustomerSegment.PREMIUM_SEGMENT,
        size=400,
        price_elasticity=0.2,
        quality_sensitivity=0.8,
        convenience_sensitivity=0.7,
        sustainability_sensitivity=0.6,
        current_market_share=0.10,
        growth_potential=0.6,
    ),
]

for segment in segments:
    market_shaper.add_market_segment(segment)

print(f"Market segments added: {len(market_shaper.market_segments)}")
for segment in market_shaper.market_segments:
    print(
        f"  {segment.segment_id.replace('_', ' ').title()}: "
        f"{segment.size:,} customers, "
        f"{segment.current_market_share:.1%} segment share"
    )

interventions = [
    StrategicIntervention(
        intervention_id="price_optimization",
        intervention_type="pricing",
        target_segments=["price_sensitive"],
        investment_cost=50000,
        expected_impact={"market_share": 0.05, "customer_satisfaction": 0.1},
        implementation_time=3,
        success_probability=0.8,
    ),
    StrategicIntervention(
        intervention_id="quality_improvement",
        intervention_type="service",
        target_segments=["quality_focused", "premium_segment"],
        investment_cost=80000,
        expected_impact={"market_share": 0.08, "brand_perception": 0.15},
        implementation_time=6,
        success_probability=0.7,
    ),
    StrategicIntervention(
        intervention_id="convenience_enhancement",
        intervention_type="technology",
        target_segments=["convenience_seeking"],
        investment_cost=60000,
        expected_impact={"market_share": 0.06, "service_speed": 0.2},
        implementation_time=4,
        success_probability=0.75,
    ),
    StrategicIntervention(
        intervention_id="sustainability_program",
        intervention_type="marketing",
        target_segments=["sustainability_conscious", "premium_segment"],
        investment_cost=40000,
        expected_impact={"market_share": 0.04, "brand_loyalty": 0.12},
        implementation_time=2,
        success_probability=0.85,
    ),
    StrategicIntervention(
        intervention_id="market_expansion",
        intervention_type="partnership",
        target_segments=["price_sensitive", "convenience_seeking"],
        investment_cost=70000,
        expected_impact={"market_share": 0.07, "market_reach": 0.25},
        implementation_time=8,
        success_probability=0.6,
    ),
]

for intervention in interventions:
    market_shaper.add_intervention(intervention)

print(f"\nStrategic interventions added: {len(market_shaper.interventions)}")
for intervention in market_shaper.interventions:
    print(
        f"  {intervention.intervention_id.replace('_', ' ').title()}: "
        f"${intervention.investment_cost:,.0f}, "
        f"targets {len(intervention.target_segments)} segment(s)"
    )

network_config = {
    "facilities": 5,
    "service_level": 0.95,
    "coverage_area": 100,
    "technology_level": "advanced",
    "sustainability_score": 0.8,
}
market_shaper.configure_network(network_config)
print(f"\nStored network configuration: {network_config}")

## Analyze opportunities and enumerate strategies

The opportunity summary uses segment sizes to express growth potential in customer equivalents and to calculate a size-weighted current market position. The strategy search then compares feasible combinations with the no-intervention baseline. Each candidate receives one stochastic simulation, so its score is a simulated estimate rather than a deterministic forecast.

In [ ]:
print("Analyzing market opportunities...")
print("=" * 40)

opportunities = market_shaper.analyze_market_opportunities()
print(f"Total market size: {opportunities['market_size']:,} customers")
print(
    "Size-weighted current market position: "
    f"{opportunities['current_market_position']:.1%}"
)
print(
    "Growth potential: "
    f"{opportunities['growth_potential']:.0f} customer equivalents"
)

print("\nHigh-growth segments:")
for segment in opportunities["high_growth_segments"]:
    print(
        f"  {segment['segment_id'].replace('_', ' ').title()}: "
        f"{segment['growth_potential']:.1%} growth potential, "
        f"{segment['size']:,} customers"
    )

print("\nUnderserved segments:")
for segment in opportunities["underserved_segments"]:
    print(
        f"  {segment['segment_id'].replace('_', ' ').title()}: "
        f"{segment['current_share']:.1%} share, "
        f"{segment['size']:,} customers"
    )

print("\nModeled competitive advantages:")
for advantage in opportunities["competitive_advantages"]:
    print(f"  - {advantage.replace('_', ' ').title()}")

print("\nStrategic priorities:")
for priority in opportunities["strategic_priorities"]:
    print(f"  - {priority.replace('_', ' ').title()}")

budget = 200000
time_horizon = 12
optimal_strategy = market_shaper.optimize_market_strategy(budget, time_horizon)
simulation_result = optimal_strategy["simulation_result"]

print("\nStrategy search results:")
print(f"  Selected interventions: {len(optimal_strategy['selected_interventions'])}")
print(f"  Total cost: ${optimal_strategy['total_cost']:,.0f}")
print(f"  Strategy score: {optimal_strategy['expected_roi']:.4f}")
print(f"  Simulated market growth: {optimal_strategy['market_impact']:.2%}")

if optimal_strategy["selected_interventions"]:
    print("\nSelected interventions:")
    for intervention in optimal_strategy["selected_interventions"]:
        print(
            f"  - {intervention.intervention_id.replace('_', ' ').title()}: "
            f"${intervention.investment_cost:,.0f}"
        )
        print(f"    Targets: {', '.join(intervention.target_segments)}")
        print(f"    Implementation time: {intervention.implementation_time} months")
else:
    print("\nThe no-intervention baseline has the highest score in this simulation.")

print("\nSimulation summary:")
print(f"  Time horizon: {simulation_result['time_horizon']} months")
print(
    "  Interventions successful in at least one active month: "
    f"{len(simulation_result['successful_interventions'])}"
)
print(f"  Approximate ROI under the stated value assumption: {simulation_result['total_roi']:.4f}")
print(f"  Customer-equivalent market growth: {simulation_result['market_growth']:.2%}")

print("\nFinal market share by segment:")
for segment in market_shaper.market_segments:
    final_share = simulation_result["final_market_share"][segment.segment_id]
    change = final_share - segment.current_market_share
    print(
        f"  {segment.segment_id.replace('_', ' ').title()}: "
        f"{final_share:.2%} ({change:+.2%})"
    )

market_report = market_shaper.generate_market_report()
print("\nMarket report summary:")
print(f"  Segments: {market_report['market_overview']['total_segments']}")
print(f"  Total market size: {market_report['market_overview']['total_market_size']:,}")
print(
    "  Growth potential: "
    f"{market_report['market_overview']['growth_potential']:.0f} "
    "customer equivalents"
)
print(
    "  Size-weighted current market position: "
    f"{market_report['market_overview']['current_market_position']:.1%}"
)

## Visualize and interpret the simulation

The first panel compares segment sizes, captured customer equivalents, and growth-potential equivalents on a common customer scale. The intervention panel displays costs and supplied market-share impacts on separate axes because they have different units. The time-series panel reports end-of-month shares. These visualizations describe the model's inputs and simulated outcomes; they do not establish causal effects in a real market.

In [ ]:
def visualize_market_shaping(
    market_shaper: MarketShaper,
    opportunities: Dict[str, Any],
    optimal_strategy: Dict[str, Any],
    simulation_result: Dict[str, Any],
    budget: float,
) -> Dict[str, Any]:
    """Plot market inputs and outcomes, then print a concise interpretation."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(16, 12)
    )

    segments = market_shaper.market_segments
    segment_names = [segment.segment_id for segment in segments]
    segment_sizes = [segment.size for segment in segments]
    captured_customers = [
        segment.size * segment.current_market_share for segment in segments
    ]
    growth_equivalents = [
        segment.size * segment.growth_potential for segment in segments
    ]

    x_positions = np.arange(len(segment_names))
    bar_width = 0.25
    ax1.bar(
        x_positions - bar_width,
        segment_sizes,
        bar_width,
        label="Segment size",
        alpha=0.8,
    )
    ax1.bar(
        x_positions,
        captured_customers,
        bar_width,
        label="Current captured customer equivalents",
        alpha=0.8,
    )
    ax1.bar(
        x_positions + bar_width,
        growth_equivalents,
        bar_width,
        label="Growth-potential equivalents",
        alpha=0.8,
    )
    ax1.set_title("Market Segment Overview")
    ax1.set_xlabel("Market segment")
    ax1.set_ylabel("Customers or customer equivalents")
    ax1.set_xticks(x_positions)
    ax1.set_xticklabels(
        [name.replace("_", " ").title() for name in segment_names],
        rotation=35,
        ha="right",
    )
    ax1.legend(fontsize=8)
    ax1.grid(axis="y", alpha=0.3)

    interventions = market_shaper.interventions
    intervention_names = [
        intervention.intervention_id.replace("_", " ").title()
        for intervention in interventions
    ]
    intervention_costs = [
        intervention.investment_cost for intervention in interventions
    ]
    market_share_impacts = [
        intervention.expected_impact.get("market_share", 0.0)
        for intervention in interventions
    ]
    intervention_positions = np.arange(len(interventions))
    cost_bars = ax2.bar(
        intervention_positions,
        intervention_costs,
        color="tab:orange",
        alpha=0.8,
        label="Investment cost",
    )
    ax2.set_title("Intervention Cost and Supplied Impact")
    ax2.set_xlabel("Intervention")
    ax2.set_ylabel("Investment cost (monetary units)")
    ax2.set_xticks(intervention_positions)
    ax2.set_xticklabels(intervention_names, rotation=35, ha="right")
    ax2.grid(axis="y", alpha=0.3)
    impact_axis = ax2.twinx()
    impact_line = impact_axis.plot(
        intervention_positions,
        market_share_impacts,
        color="tab:blue",
        marker="o",
        linewidth=2,
        label="Supplied market-share impact",
    )
    impact_axis.set_ylabel("Market-share impact input (scaled in simulation)")
    ax2.legend(
        [cost_bars, impact_line[0]],
        ["Investment cost", "Supplied market-share impact"],
        loc="upper left",
        fontsize=8,
    )

    months = [
        response["month"] for response in simulation_result["monthly_responses"]
    ]
    for segment in segments:
        share_evolution = [
            response["market_shares"].get(segment.segment_id, 0.0)
            for response in simulation_result["monthly_responses"]
        ]
        ax3.plot(
            months,
            share_evolution,
            marker="o",
            label=segment.segment_id.replace("_", " ").title(),
        )
    ax3.set_title("Segment Market Share by Month")
    ax3.set_xlabel("Month")
    ax3.set_ylabel("Market share within segment")
    ax3.set_ylim(0.0, 1.0)
    ax3.legend(fontsize=8)
    ax3.grid(alpha=0.3)

    selected_interventions = optimal_strategy["selected_interventions"]
    selected_count = len(selected_interventions)
    success_rate = (
        len(simulation_result["successful_interventions"]) / selected_count * 100
        if selected_count > 0
        else 0.0
    )
    budget_utilization = (
        optimal_strategy["total_cost"] / budget * 100 if budget > 0 else 0.0
    )
    impact_metrics = {
        "Market growth (%)": simulation_result["market_growth"] * 100,
        "Approximate ROI (%)": simulation_result["total_roi"] * 100,
        "Selected interventions with success (%)": success_rate,
        "Budget used (%)": budget_utilization,
    }
    metric_names = list(impact_metrics)
    metric_values = list(impact_metrics.values())
    metric_bars = ax4.bar(metric_names, metric_values, alpha=0.8)
    ax4.set_title("Simulation Indicators")
    ax4.set_ylabel("Percentage (%)")
    ax4.set_xticks(np.arange(len(metric_names)))
    ax4.set_xticklabels(metric_names, rotation=25, ha="right")
    ax4.grid(axis="y", alpha=0.3)
    for bar, value in zip(metric_bars, metric_values):
        ax4.annotate(
            f"{value:.1f}%",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
        )

    plt.tight_layout()
    plt.show()

    initial_customer_equivalents = sum(
        segment.size * segment.current_market_share for segment in segments
    )
    final_customer_equivalents = sum(
        segment.size * simulation_result["final_market_share"][segment.segment_id]
        for segment in segments
    )

    print("\nSimulation interpretation:")
    print("=" * 30)
    print(
        "  Initial customer equivalents: "
        f"{initial_customer_equivalents:.1f}"
    )
    print(
        "  Final customer equivalents: "
        f"{final_customer_equivalents:.1f}"
    )
    print(
        "  Change in customer equivalents: "
        f"{final_customer_equivalents - initial_customer_equivalents:+.1f}"
    )
    print(f"  Approximate ROI under the stated value assumption: {simulation_result['total_roi']:.4f}")
    print(f"  Customer-equivalent market growth: {simulation_result['market_growth']:.2%}")
    print(
        "  Budget utilization: "
        f"{optimal_strategy['total_cost'] / budget:.1%}"
        if budget > 0
        else "  Budget utilization: not defined for a zero budget"
    )

    return simulation_result


visualize_market_shaping(
    market_shaper,
    opportunities,
    optimal_strategy,
    simulation_result,
    budget,
)

## Strengths, limitations, and use

### Strengths

- Segment-level inputs make it possible to target interventions rather than treating all customers as identical.
- Budget-feasible combinations are evaluated explicitly, and the no-intervention baseline prevents the model from requiring a strategy that scores worse than doing nothing.
- The stochastic simulation makes implementation uncertainty visible, while a fixed random seed supports reproducible notebook runs.
- The code separates segment analysis, simulation, combination generation, optimization, and reporting into inspectable steps.

### Limitations

- The market-share impacts, success probabilities, and monetary value per additional customer are assumptions. Results should not be interpreted as validated forecasts or financial recommendations.
- One simulation per candidate can make the selected combination sensitive to the random outcomes in that run. The score is not an estimate with a reported uncertainty interval.
- Segment shares are modeled independently, so their sum need not equal one. The model does not represent competitor market shares or transfers between segments.
- Network configuration and segment elasticities do not determine intervention impacts in the simulation. This example therefore illustrates intervention selection and response simulation, not a fully integrated demand-and-network-design model.
- The search is restricted to combinations of at most four interventions, and its exhaustive enumeration becomes more costly as the candidate set grows.

### When this model is useful

This structure can support early strategic discussions when an organization wants to compare a small set of interventions, make budget limits explicit, and inspect how uncertain implementation outcomes affect modeled market shares. Before operational use, the assumptions would need to be calibrated with appropriate data, and the model's scope should be extended only when the decision context requires it.

### Interpretation

A selected strategy is the highest-scoring feasible combination among those enumerated for the specified simulation run and assumptions. It is not evidence that the interventions will cause the simulated changes in a real market. A no-intervention result means that, under the sampled outcomes and the model's score, no evaluated feasible combination exceeded the baseline.